In [1]:
import json

In [2]:
from quenchpath.chemistry.network import VENOT2019
from quenchpath.chemistry.polynomials import (
    find_duplicate_species,
    parse_nasa_polynomials,
    species_name_counts,
    validate_network_coverage,
)
from quenchpath.mypaths import path_to_networks_processed, path_to_raw_networks

In [3]:
# Read the 7 term NASA polynomials
fname = "coeff_NASA_sc.dat"  # in ATMO coeff_NASA_sc.dat
with open(path_to_raw_networks / fname) as f:
    raw_lines = f.read().splitlines()

# What: this notebook is a thin driver - read the raw file, call the
# library functions, save the result.
# Why: parsing/validation logic lives in quenchpath.chemistry.polynomials.
plns = parse_nasa_polynomials(raw_lines)

# Note: one of SO2 sets of polynomials has 15 parameters instead of 14

In [4]:
# What: check for duplicate species in .dat.
# Why: parse_nasa_polynomials() keeps only the FIRST entry for a
# duplicated species (matching the UM's own reference reader) -
# confirmed this makes no difference to any species actually used by
# VENOT2019 as it currently stands.
duplicate_species = find_duplicate_species(raw_lines)
counts = species_name_counts(raw_lines)
# Print duplicate species names and counts in .dat
[(species, counts[species]) for species in duplicate_species]

[('C6H10', 2),
 ('NCN', 2),
 ('HON', 2),
 ('NH2OH', 2),
 ('NaCl', 2),
 ('KCl', 2),
 ('KO', 2),
 ('SO', 2),
 ('SO2', 3),
 ('S2', 2),
 ('ClO', 3),
 ('NaO', 2),
 ('NaOH', 2),
 ('Ca', 2)]

In [5]:
n_entries = len(raw_lines) // 3
print(f"Number of species entries in .dat: {n_entries}")
print(f"Number of individual species: {len(plns)}")
print(f"Number of duplicate species entries in .dat: {n_entries - len(plns)}")
print(f"Number of duplicate species: {len(duplicate_species)}")  # SO2 and ClO have 3 duplicates

Number of species entries in .dat: 399
Number of individual species: 383
Number of duplicate species entries in .dat: 16
Number of duplicate species: 14


In [6]:
# What: choose a chemical network.
# Why: to check whether all polynomials data are available for
# calculating reaction rates.
NETWORK = VENOT2019

In [7]:
# Check that every network species has exactly one (non-ambiguous) set of
# 7-term NASA polynomials available - validate_network_coverage() raises
# ValueError otherwise (duplicate or missing species).
validate_network_coverage(plns, duplicate_species, NETWORK["all_gases"])

In [8]:
# Save the 7 term NASA polynomials as a json file
with open(path_to_networks_processed / "nasa_polynomials_7_term.json", "w") as fpolynomials:
    json.dump(plns, fpolynomials)